# GP03 Public Adapter

Verify the public test adapter against the selected nine tasks.

This source copy comes from **GP03_Public_Adapter_Acceptance_v1_0.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

**Clear-text source release:** No embedded ZIP, Base64 or hex payloads are required to read the notebook source.


### Preserved execution evidence — source archive

- **Executed source:** `Paper/GP03/gp03-2.ipynb` in the author-provided `Paper (2)(2).zip` (10 November 2026).
- **Saved results:** 6 human-readable Jupyter output records restored from matching executed cells.
- **Source fidelity:** 9 identical code cells; 0 proven value-equivalent readable source strings; 0 historical configuration cell(s) restored to align with archived execution.
- **Excluded display artifacts:** 2 active, encoded, or binary browser-download outputs were omitted; the source archive retains them. These are *not* replacements for the sealed experiment ZIPs.
- **Reproduction distinction:** Saved historical output does **not** mean this notebook has been rerun in the current environment. The original experiment may require Kaggle mounts, GPUs, offline wheels and authorised archives. The additional execution guide above details the prerequisites.


## How to execute or reproduce this implementation stage

**Execution mode:** CPU.  
**Prerequisites:** Selected nine GP03 task snapshots and pinned public pytest selectors.

**Steps:** Run adapter qualification in fresh workspaces and check exact node collection and public check provenance for each task.

**Reproduction boundary:** Conditional: official task snapshots and correct wheel environment required.

> **Historical evidence:** This release retains implementation source but removes historical cell outputs. Results from stochastic GPU runs are not guaranteed to repeat byte-for-byte. Use the sealed evidence records and the E17 reviewer companion to reconstruct accepted historical outcomes.


# GP03 public adapter acceptance

Validate the public pytest adapter on the nine proposed held-out tasks. Each task runs its existing public check once on a private copy of the published baseline. The same callable is intended for both evaluation arms. The notebook uses CPU, with zero model calls and zero protected tests.

1. Import this notebook into Kaggle. Set **Accelerator** to **None** and **Internet** to **Off**. Use the same CPU image that ran GP03 qualification v1.1.
2. Add the **Gemma 4 Developer Agent** competition input. Keep `GP03_CPU_Qualification_20261108T225859Z_ca48fb6a.zip` in `/kaggle/working` or add it as a mounted input. Keep `CPU_EVIDENCE_ZIP = None` if the original filename is visible at either location.
3. Choose **Run All**. Expect a timed row for each task and `GP03_PUBLIC_ADAPTER_ACCEPTED` for 9/9 accepted. The final cell offers a ZIP download and shows the file link.

Acceptance requires the same source, test definitions, node IDs, JUnit counts and outcome as the qualification receipts. The adapter checks source isolation and preserves internal certificate links. This is one prerequisite for freezing the cohort and the paired execution configuration.


## 1. CPU settings

In [1]:
from collections import Counter
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path, PurePosixPath
from typing import Mapping, Sequence
import ast
import csv
import fcntl
import hashlib
import json
import os
import platform
import posixpath
import re
import shutil
import signal
import subprocess
import sys
import tarfile
import tempfile
import time
import uuid
import xml.etree.ElementTree as ET
import zipfile
from dataclasses import asdict
from typing import Callable

COMPETITION_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
WORK_DIR = Path("/kaggle/working")
CPU_EVIDENCE_ZIP = None  # Set an absolute path only if the original ZIP has been renamed.
AUTO_DOWNLOAD_ZIP = True

## 2. Qualification evidence and proposed pair order

In [2]:
ADAPTER_RECIPE_SHA256 = 'ac01106525c4d97767b950cf565de028526f9934e93375e2d6ab981b0863bbca'

ELIGIBLE_POOL_SHA256 = 'cba7da409788164e4f559f74d325088333c033db57b2739adf8b86955d68b541'

GP03_CPU_EVIDENCE = {'cpu_environment_sha256': '0f53e69268c9834ce589f44b8adf8a27263e8daab2ff90f5e37a333ffa34d772',
 'filename': 'GP03_CPU_Qualification_20261108T225859Z_ca48fb6a.zip',
 'gp00_split_sha256': '27b50b7f46c510c55b7b7bb264aba729abba0cc9e53647e596690c94cd79bb09',
 'notebook_recipe_sha256': '0ae4bc06d88271e9b2ccd3c4fcdae182d9b58b07743eb7130cf7daa317a0f750',
 'qualification_run_sha256': '085458dbf8f424ea23dcc57914e4c284c1bcaaf55ac1f8e5c56d678c8717cff9',
 'sha256': '166d33d3e2426c26831b8473c2098ffc6cf990072165d492fe4926784b03cce2',
 'tasks_sha256': 'e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6'}

GP03_PROPOSED_PAIRS = [{'pair_index': 1,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'fastapi_14964',
  'repo': 'fastapi/fastapi',
  'base_commit': 'bc06e4296d588f39f38c7ac2fc96d3913c565962',
  'snapshot_bytes': 271065089,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.0_retained',
  'cpu_check_member': 'tasks/fastapi_14964/CPU_Check_Manifest.json',
  'cpu_check_sha256': '30a05a807d13a9cbf8cb03f6a13f8b55e5c0ea8e5b5194654d8fe9507eac743d',
  'cpu_receipt_sha256': 'f5aebd1a3823212bb17b7e06b00ef59ef1f594c8de86f4cc82796a923d06e783',
  'snapshot_sha256': 'f09059d29d925ac0b5ed96ddd8ee164e656cdf164d8ae88a53bae3ff783659e8',
  'native_head': '47da909b715240cfe7dc2b04d2994c56c9153b52',
  'native_tree': 'cac1a745f81c2ac504ce548da1a4106cc53a5e23',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_orjson_response_class.py',
                        '-q',
                        '--tb=short'],
  'expected_collected': 1,
  'has_snapshot_links': False,
  'rank_sha256': '0b1c850408e946a63c547c1ae99ce1cfad9505a10ae7ebf51960f3a725266f75'},
 {'pair_index': 2,
  'seed': 42,
  'arm_order': ['C', 'B'],
  'task_id': 'requests_7502',
  'repo': 'psf/requests',
  'base_commit': '661970d171d9c3e12e4c789c4768db647d8c4da0',
  'snapshot_bytes': 37463957,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/requests_7502/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'c6a3ae1855099875f96cfa57f9b5f31a0007b6f7699d8ec3aec2fa064dd2a2cc',
  'cpu_receipt_sha256': '93edc64af99aaf183eed4eb79a0198c5b4d646cfd1c57bca0423c72edf2ac57a',
  'snapshot_sha256': 'a7da726a17f54ea4d43d1a21e3e9f3e95bfa2787a845a18e2d30a88fcc1f53fb',
  'native_head': 'ced47d0b985f67110197b488df11f54e4d355eca',
  'native_tree': 'ea72ddd13e602fe8f9d9a5546d1be5f64b46046c',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_requests.py::TestRequests::test_basicauth_encodes_byte_strings',
                        '-q',
                        '--tb=short'],
  'expected_collected': 1,
  'has_snapshot_links': True,
  'rank_sha256': '04e9442de4385c144ad4dfa09f1633861723f1211563255f7b9f38956450f8a4'},
 {'pair_index': 3,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'rich_3718',
  'repo': 'Textualize/rich',
  'base_commit': '7d36119f8378b36129502c3281a7faffef837e02',
  'snapshot_bytes': 95464718,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/rich_3718/CPU_Check_Manifest.json',
  'cpu_check_sha256': '7fad2c43829f54b41af538bda13a4c1ad54d06c4e18aac0b40df212598734cb1',
  'cpu_receipt_sha256': 'dabf2c77ced00da25bdeceb8a876510e67295d1ec2e7c7c7c1fc730a04e4037c',
  'snapshot_sha256': 'f66ac24cfee77115da4b15dfad5e5adbab07faedb53a62693bf16b75c2750eb6',
  'native_head': 'fef719c6fa6ea0305898a10ca7244b9da200a6cb',
  'native_tree': 'e1652a57c5e0929cc91da1f8333834d9ae898515',
  'public_check_argv': ['/usr/bin/python3', '-m', 'pytest', 'tests/test_panel.py', '-q', '--tb=short'],
  'expected_collected': 11,
  'has_snapshot_links': False,
  'rank_sha256': '2a13e31c5f90e29d0762c259ff078c17571643f6db97b63f6ade61ef1cf500bf'},
 {'pair_index': 4,
  'seed': 42,
  'arm_order': ['C', 'B'],
  'task_id': 'fastapi_12942',
  'repo': 'fastapi/fastapi',
  'base_commit': '27c0f7e75fc0ce26ca75295d455eb36f0ca033d3',
  'snapshot_bytes': 218214164,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.0_retained',
  'cpu_check_member': 'tasks/fastapi_12942/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'b2ceca01de32d22080a3e5ae9b7f6772bc15d5f3871e0535142870e3e9b8efb8',
  'cpu_receipt_sha256': 'a9f795e8af84f2d11e8bc8d70bc747d5d2166ee8e81747790561dd72cc9e56ab',
  'snapshot_sha256': '68f9786237a6b5cc811190e95f8d45045d850aa09f77cb8be71e6bfd8e900a2a',
  'native_head': '8c426cdec82445a3f69c8af6ae991ab26d10ea80',
  'native_tree': '09e60c5d118d3d01fdcc6fdbb970c4b44ef22451',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_response_model_as_return_annotation.py',
                        '-q',
                        '--tb=short'],
  'expected_collected': 38,
  'has_snapshot_links': False,
  'rank_sha256': '0c84af5514ba9d8d279adc9713c0966642d5978893e576561125dcf276821336'},
 {'pair_index': 5,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'requests_7205',
  'repo': 'psf/requests',
  'base_commit': '1b40fdd004bfc8ba5301bcf8a6908264e9b6b877',
  'snapshot_bytes': 37402474,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/requests_7205/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'ac5b3accf4b666a01fe937fe459a4146d1f9e0c8fffe9eeee416aba8a293039b',
  'cpu_receipt_sha256': '7285909ade50c8e6533cb3eb87e640a99141b6c7f6db8d04006b5b155bdde251',
  'snapshot_sha256': '42dc3d360f842a5c7a66d0008ac05447fcd807dc33ee8cc18d2f2d4876283106',
  'native_head': '443f0b73558781b7dac0e304ad268a30ee35aaeb',
  'native_tree': '683d44befe0d3519469011c35e46f332eebbc010',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_requests.py::TestRequests::test_proxy_auth_empty_pass',
                        '-q',
                        '--tb=short'],
  'expected_collected': 1,
  'has_snapshot_links': True,
  'rank_sha256': '0a3fd9629109362be6725a7ea800d14b08d4904a587634ec0494501afa5aa08c'},
 {'pair_index': 6,
  'seed': 42,
  'arm_order': ['C', 'B'],
  'task_id': 'rich_4077',
  'repo': 'Textualize/rich',
  'base_commit': '58ac15120fe254886c10d6253667733c83604cb4',
  'snapshot_bytes': 97076310,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/rich_4077/CPU_Check_Manifest.json',
  'cpu_check_sha256': '202833bc7ea98a1ff967c2480884d15d67bae2adcf682302c58af265f2fde382',
  'cpu_receipt_sha256': '58d6eb45f64a93c28491f55510d6df8b4a4f744d0dbc5571fe80de4d8ee58e8e',
  'snapshot_sha256': '96aab6cf7a51b313a5e203b27c7bc4c557b044eaa65f0120fe0c27357fa6ed4f',
  'native_head': '11cdcf5c9e81f42d4e9b2e3981bc8f67ed44b18e',
  'native_tree': 'bc6ede75faeaee74089fa8f3b91fa5f5e3ac3138',
  'public_check_argv': ['/usr/bin/python3', '-m', 'pytest', 'tests/test_file_proxy.py', '-q', '--tb=short'],
  'expected_collected': 3,
  'has_snapshot_links': False,
  'rank_sha256': '3f0898754663f089d61c6483eda14c56ec4d04eeebda696f098e0b116d1cc39e'},
 {'pair_index': 7,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'fastapi_15023',
  'repo': 'fastapi/fastapi',
  'base_commit': 'c0836dc1b7f4ba91e9563c8c4ccd0ebe655bc914',
  'snapshot_bytes': 275792435,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.0_retained',
  'cpu_check_member': 'tasks/fastapi_15023/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'a0924d89fb59c4f42653c5d6a52a0a102faf5590ad18cdb322b627f1c907d47b',
  'cpu_receipt_sha256': '67d5a51c6a0800c841c6edf7350fd275aef0311e05ef1bf243a6b124fb264609',
  'snapshot_sha256': 'b3873cfe6c959e13ef6d655648ab1cf4931d096af69debf70550ecfdf4146ccf',
  'native_head': 'f7f72e123b1f425f7aa98a13920a3413856077cf',
  'native_tree': 'e677615e75916874f814d2b2499f6efbb22aa00a',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_dependency_after_yield_streaming.py',
                        '-q',
                        '--tb=short'],
  'expected_collected': 7,
  'has_snapshot_links': False,
  'rank_sha256': '1f0a7967c5bceccdcc4f9c8262d571fb6cab317ee2dd029039a33bc8fefc39f9'},
 {'pair_index': 8,
  'seed': 42,
  'arm_order': ['C', 'B'],
  'task_id': 'requests_7427',
  'repo': 'psf/requests',
  'base_commit': 'b684dcb9bbf3aa557d1238e72062c4a29737dd1c',
  'snapshot_bytes': 37176965,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/requests_7427/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'fecce775e5605870467133259043c81e60ea0039319ddbd30b2f80860b8a2db8',
  'cpu_receipt_sha256': 'c3017eaa1270aaf077bdedd5e1f256d6ccc486dbc475e84fa38a8397b3528988',
  'snapshot_sha256': 'd8f2092187969feb4f4b32b501682d76c09e27173371b4673c1daceef72f2e04',
  'native_head': '6cd34441464a253437ac6d7846379fd4f945b97b',
  'native_tree': 'f89ba9281369ca35eaad0f27853941e129836be2',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_requests.py::TestRequests::test_proxy_authorization_not_appended_to_https_request',
                        '-q',
                        '--tb=short'],
  'expected_collected': 2,
  'has_snapshot_links': True,
  'rank_sha256': '0a4bb2e29eee1a6c909d019abe71822ac222153ad3672536c87c1b9982f364f5'},
 {'pair_index': 9,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'rich_3043',
  'repo': 'Textualize/rich',
  'base_commit': '8a616de8e25db3c93abaaa220c493d80878de69d',
  'snapshot_bytes': 90818810,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/rich_3043/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'ec7ff0096a38d1d9999f65c3f6ad0e303597365593e5287b691c514b0c13eba9',
  'cpu_receipt_sha256': '113364c86485926628f62eb4f50fcd7bf753686959218b25da606cf3dd05615b',
  'snapshot_sha256': '71edf847e2907fcde33ce124f6d57871448c8b55aba059cda75bad05b7653d2d',
  'native_head': 'c41239b93ac0e9af2f4e3890dfb807f942b31f89',
  'native_tree': 'c2ab870ce396f066664ea9de5678ee71ca24dd92',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_console.py::test_export_html',
                        '-q',
                        '--tb=short'],
  'expected_collected': 1,
  'has_snapshot_links': False,
  'rank_sha256': '4b5b2a65bb1e9464b6570821066d4291aeb6dd7e4be135c3d7ff298317bb869e'}]

GP03_DEVELOPMENT = [
    ('fastapi_15589', 'fastapi/fastapi', '22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b'),
    ('rich_4079', 'Textualize/rich', '19c67b9a3479841e9133bea94607c89ee931d3fc'),
    ('fastapi_14479', 'fastapi/fastapi', '2b212ddd7604891d42e2ffbafe764c334230bdb7'),
    ('fastapi_5624', 'fastapi/fastapi', 'c6487ed632056e450d844846a1b63be551a3cbc6'),
    ('httpx_3672', 'encode/httpx', '4acf5c2c37714cc63b5cf71b3e284fca83c90311'),
    ('requests_6757', 'psf/requests', '0e322af87745eff34caffe4df68456ebc20d9068'),
    ('rich_2943', 'Textualize/rich', '610fd75c433c3dfcd9c296e8b91363979127b9fa'),
    ('rich_3130', 'Textualize/rich', '720800e6930d85ad027b1e9bd0cbb96b5e994ce3'),
]

## 3. CPU runtime and process helpers

Reused from the pinned GP03 qualification notebook.

In [3]:
POLICY = {
    "version": "GP03_CPU_Qualification_v1.1",
    "selector_rule": "public_issue_to_tracked_test_paths_and_symbols_v1",
    "max_selectors": 5,
    "large_file_test_functions": 50,
    "max_collected_nodes": 200,
    "snapshot_bytes_expanded_cap": 6 * 1024**3,
    "task_wall_seconds": 300,
    "collect_wall_seconds": 55,
    "baseline_wall_seconds": 40,
    "final_reserve_seconds": 60,
    "optional_import_fallback": [
        "dirty_equals", "email_validator", "httpbin", "inline_snapshot", "multipart", "orjson",
        "pytest_asyncio", "pytest_httpbin", "pytest_mock", "trustme", "ujson", "uvicorn",
    ],
    "optional_fixture_fallback": ["httpbin", "httpbin_secure"],
}

PACKAGE_LAYOUTS = {
    "fastapi/fastapi": ("fastapi", ("fastapi", "src/fastapi")),
    "Textualize/rich": ("rich", ("rich", "src/rich")),
    "psf/requests": ("requests", ("requests", "src/requests")),
}

EXTRA_DEPENDENCIES = (
    "attrs", "brotli", "charset-normalizer", "cryptography", "dirty-equals",
    "flask", "httpbin", "pytest-httpbin", "pytest-mock", "requests-mock",
    "simplejson", "socksio", "trustme", "uvicorn", "werkzeug",
)

OFFLINE_FLAGS = {
    "CUDA_VISIBLE_DEVICES": "",
    "GIT_CONFIG_NOSYSTEM": "1",
    "GIT_CONFIG_GLOBAL": os.devnull,
    "GIT_OPTIONAL_LOCKS": "0",
    "GIT_TERMINAL_PROMPT": "0",
    "HF_HUB_OFFLINE": "1",
    "PIP_CONFIG_FILE": os.devnull,
    "PIP_NO_INDEX": "1",
    "PIP_NO_INPUT": "1",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONHASHSEED": "0",
    "PYTHONNOUSERSITE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}

PUBLIC_TEST = re.compile(r"tests/(?:[A-Za-z0-9_-]+/)*[A-Za-z0-9_-]+\.py")

VALID_ID = re.compile(r"[A-Za-z0-9_]+")

GIT_ID = re.compile(r"[0-9a-f]{40}")

class CPUHold(RuntimeError):
    def __init__(self, reason, detail):
        super().__init__(detail)
        self.reason = reason

TEST_DEPENDENCIES = (
    'pytest', 'pluggy', 'iniconfig', 'packaging', 'pytest-asyncio', 'pytest-cov',
    'pytest-timeout', 'coverage', 'typing-extensions', 'pydantic', 'pydantic-core',
    'typing-inspection', 'annotated-types', 'annotated-doc', 'starlette', 'anyio',
    'httpx', 'httpcore', 'h11', 'h2', 'hpack', 'hyperframe', 'idna', 'sniffio',
    'certifi', 'requests', 'urllib3', 'python-multipart', 'email-validator',
    'markdown-it-py', 'mdurl', 'pygments', 'exceptiongroup', 'trio',
    'fastapi-cli', 'jinja2', 'orjson', 'ujson', 'pyyaml',
)

def canonical_hash(obj: object) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=True,
                                      separators=(",", ":"), default=str).encode()).hexdigest()

def environment_fingerprint(config: Mapping[str, object]) -> str:
    """Hash explicit environment identity supplied by the trusted harness.

    In live integration, include installed wheel/source hashes and relevant runtime
    settings, not just package names, and refresh after environment modifications.
    """
    if not config:
        raise ValueError("Nonempty trusted environment manifest required")
    return canonical_hash(dict(config))

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(262144), b""):
            h.update(block)
    return h.hexdigest()

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(data, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    temporary.replace(path)

def _cancel_process_group(proc: subprocess.Popen, grace: float = 0.15) -> None:
    """Terminate a process GROUP even if its leader has already exited.

    A parent exiting after SIGTERM does not prove children have exited. All
    checks run in their own POSIX session. After a brief grace, SIGKILL is sent
    to the *group*, including remaining descendants, even if leader.poll() != None.
    """
    if os.name == "posix":
        pgid = proc.pid  # subprocess.Popen(start_new_session=True)
        try:
            os.killpg(pgid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=max(grace, 0.02))
        except subprocess.TimeoutExpired:
            pass
        # Crucial: don't return merely because the group LEADER exited.
        # An orphan/daemonized child in this process group may still be alive.
        time.sleep(min(0.04, max(grace, 0.0)))
        try:
            os.killpg(pgid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=2)
        except subprocess.TimeoutExpired:
            pass
    else:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=grace)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=2)

def load_gp03_inputs(competition, candidates, development, release_sha):
    """Use issue text and metadata from the pinned release, with an explicit projection."""
    if not (competition / "tasks.jsonl").is_file():
        raise CPUHold("TASK_RELEASE_MISSING", f"Mount the Developer Agent input at {competition}")
    if not (competition / "snapshots").is_dir():
        raise CPUHold("SNAPSHOT_MOUNT_MISSING", f"Expected {competition / 'snapshots'}")
    if sha256_file(competition / "tasks.jsonl") != release_sha:
        raise CPUHold("TASK_RELEASE_MISMATCH", "tasks.jsonl SHA-256 differs from the GP00 release")
    index = {r["task_id"]: r for r in candidates}
    groups = {(r["repo"], r["base_commit"]) for r in candidates}
    dev_ids = {r[0] for r in development}
    dev_groups = {(r[1], r[2]) for r in development}
    if len(index) != len(candidates) or len(groups) != len(candidates):
        raise CPUHold("COHORT_DUPLICATE", "Candidate IDs and repository/commit groups must be unique")
    if dev_ids.intersection(index) or dev_groups.intersection(groups):
        raise CPUHold("COHORT_OVERLAP", "Candidate set intersects the development set")
    for task, row in index.items():
        if (not VALID_ID.fullmatch(task) or row["repo"] not in PACKAGE_LAYOUTS
                or not GIT_ID.fullmatch(row["base_commit"]) or row["snapshot_bytes"] <= 0):
            raise CPUHold("COHORT_INVALID", f"Invalid public identity for {task}")

    issues = {}
    with (competition / "tasks.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            raw = json.loads(line)
            task = raw.get("instance_id")
            if task in index:
                pin = index[task]
                if task in issues or (raw.get("repo"), raw.get("base_commit")) != (
                    pin["repo"], pin["base_commit"]
                ):
                    raise CPUHold("TASK_IDENTITY_MISMATCH", f"Release identity differs for {task}")
                issue = raw.get("problem_statement")
                if not isinstance(issue, str) or not issue.strip():
                    raise CPUHold("PUBLIC_ISSUE_MISSING", f"Public issue description missing for {task}")
                issues[task] = issue
            del raw
    if set(issues) != set(index):
        raise CPUHold("CANDIDATE_MISSING_FROM_RELEASE", "A selected GP00 case is missing")
    return issues

def environment_profile():
    if os.name != "posix" or sys.version_info < (3, 12):
        raise CPUHold("RUNTIME_UNSUPPORTED", "Use the Kaggle Linux CPU image with Python 3.12 or later")
    try:
        import pytest
    except ImportError as exc:
        raise CPUHold("PYTEST_UNAVAILABLE", "Use the organizer CPU image with pytest available") from exc
    if shutil.which("git") is None:
        raise CPUHold("GIT_UNAVAILABLE", "The published snapshots require Git")
    packages = {}
    for name in sorted(set(TEST_DEPENDENCIES) | set(EXTRA_DEPENDENCIES)):
        try:
            dist = metadata.distribution(name)
        except metadata.PackageNotFoundError:
            packages[name] = None
        else:
            record = dist.read_text("RECORD") or dist.read_text("METADATA") or ""
            packages[name] = {"version": dist.version, "record_sha256": hashlib.sha256(record.encode()).hexdigest()}
    return {
        "python_version": platform.python_version(),
        "python_executable": sys.executable,
        "python_executable_sha256": sha256_file(Path(sys.executable).resolve()),
        "machine": platform.machine(),
        "pytest_version": pytest.__version__,
        "test_dependencies": packages,
        "flags": OFFLINE_FLAGS,
    }

def command_env(root, temporary, capture=None, package=None):
    home = temporary / "home"
    home.mkdir(exist_ok=True)
    paths = [root, root / "src"]
    if capture is not None:
        paths.append(temporary)
    env = {
        "PATH": os.environ.get("PATH", os.defpath), "HOME": str(home),
        "LANG": "C.UTF-8", "LC_ALL": "C.UTF-8", "TZ": "UTC",
        "TMPDIR": str(temporary), "TMP": str(temporary), "TEMP": str(temporary),
        "PYTHONPATH": os.pathsep.join(map(str, paths)), **OFFLINE_FLAGS,
    }
    if capture is not None:
        env.update(GP03_CAPTURE_PATH=str(capture), GP03_PUBLIC_PACKAGE=package)
    return env

def bounded_command(argv, root, env, output, timeout):
    output.mkdir(parents=True, exist_ok=False)
    started = time.monotonic()
    with (output / "stdout.txt").open("wb") as stdout, (output / "stderr.txt").open("wb") as stderr:
        proc = subprocess.Popen(
            argv, cwd=root, env=env, stdout=stdout, stderr=stderr, start_new_session=True,
        )
        timed_out = False
        try:
            proc.wait(timeout=max(0.01, timeout))
        except subprocess.TimeoutExpired:
            timed_out = True
        finally:
            try:
                os.killpg(proc.pid, 0)
            except ProcessLookupError:
                pass
            else:
                _cancel_process_group(proc)
    return {
        "argv": list(argv), "exit_code": proc.returncode, "timed_out": timed_out,
        "seconds": round(time.monotonic() - started, 3),
    }

def remaining(deadline, phase_cap):
    seconds = deadline - time.monotonic()
    if seconds <= 0:
        raise CPUHold("TASK_DEADLINE", "The CPU task wall clock expired")
    return min(seconds, phase_cap)

## 4. Public snapshot staging and source fingerprints

Reused from the pinned GP03 qualification notebook.

In [4]:
EXCLUDED_DIRS = frozenset({
    ".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache",
    ".ruff_cache", ".tox", ".nox", ".cache", "node_modules", ".ipynb_checkpoints",
    ".gp02c_results", "__ledger_output__", "htmlcov", ".coverage-data",
})

EXCLUDED_NAMES = frozenset({".coverage", ".DS_Store", "coverage.xml"})

EXCLUDED_SUFFIXES = frozenset({".pyc", ".pyo"})

PYTEST_CONFIG_NAMES = (
    "pytest.ini", ".pytest.ini", "pyproject.toml", "setup.cfg", "tox.ini",
    "pytest.toml", ".pytest.toml",
)

RICH_OPTIONAL_TEST_DEFINITIONS = (
    "conftest.py", "tests/conftest.py", "tests/__init__.py",
    *PYTEST_CONFIG_NAMES, *(f"tests/{name}" for name in PYTEST_CONFIG_NAMES),
)

@dataclass(frozen=True)
class Snapshot:
    sha256: str
    files: int
    entries: tuple[tuple[str, str, str], ...]

class WorkspaceFingerprinter:
    """Conservative all-files policy, except a fixed generated-artifact exclusion list.

    Includes untracked source and config; file additions, deletions and symlink
    targets affect the fingerprint. NEVER walk an evaluator/protected workspace.
    """
    def __init__(self, root: str | Path):
        self.root = Path(root).resolve(strict=True)
        if not self.root.is_dir():
            raise ValueError("Workspace root must be a directory")

    def snapshot(self) -> Snapshot:
        entries: list[tuple[str, str, str]] = []
        for base, dirs, files in os.walk(self.root, followlinks=False):
            basedir = Path(base)
            # Symlink directories are recorded as links, never dereferenced.
            for name in list(dirs):
                full = basedir / name
                if name in EXCLUDED_DIRS:
                    dirs.remove(name)
                elif full.is_symlink():
                    dirs.remove(name)
                    rel = full.relative_to(self.root).as_posix()
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
            for name in files:
                if name in EXCLUDED_NAMES or Path(name).suffix in EXCLUDED_SUFFIXES:
                    continue
                full = basedir / name
                rel = full.relative_to(self.root).as_posix()
                if full.is_symlink():
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
                elif full.is_file():
                    h = hashlib.sha256()
                    with full.open("rb") as f:
                        for block in iter(lambda: f.read(256 * 1024), b""):
                            h.update(block)
                    entries.append((rel, "F", h.hexdigest()))
        entries.sort()
        return Snapshot(canonical_hash(entries), len(entries), tuple(entries))

def definition_files_fingerprint(root: str | Path, relative_files: Sequence[str], *,
                                 allow_missing: bool = False) -> str:
    """Bind a selected visible test to its current test/config file bytes.

    File deletions, symlinks, path traversal and non-files fail closed. Caller
    supplies only approved visible test/config paths; NEVER protected tests.
    """
    root = Path(root).resolve(strict=True)
    if not relative_files or len(relative_files) != len(set(relative_files)):
        raise ValueError("Nonempty unique visible test definition paths required")
    entries = []
    for name in sorted(relative_files):
        relative = Path(name)
        if (not name or relative.is_absolute() or any(x in {"..", "."} for x in relative.parts)
            or relative.as_posix() != name):
            raise ValueError("Invalid or noncanonical visible definition path")
        path = root / relative
        if path.is_symlink() or not path.resolve().is_relative_to(root):
            raise ValueError("Visible definition unsafe or symlinked")
        if not path.is_file():
            if (allow_missing and not path.exists() and
                (name in RICH_OPTIONAL_TEST_DEFINITIONS or relative.name == "conftest.py")):
                entries.append((name, 'ABSENT_AT_FREEZE'))
                continue
            raise ValueError("Visible definition missing or unsafe")
        entries.append((name, hashlib.sha256(path.read_bytes()).hexdigest()))
    return canonical_hash(entries)

CONFIG_FILES = ('setup.py', 'setup.cfg', 'pytest.ini', '.pytest.ini', 'pytest.toml',
                '.pytest.toml', 'pyproject.toml', 'tox.ini', 'conftest.py')

HERMETIC_INI = """[pytest]
addopts = -p no:anyio
norecursedirs = .* build dist venv
python_classes = Test* *Test
python_files = test_*.py *_test.py
filterwarnings =
    ignore::DeprecationWarning
    ignore::UserWarning
"""

HERMETIC_HEADER = '# Hermetic test discovery hook for SWE-gemma\n'

def snapshot_link_target(name, linkname, symbolic):
    if not linkname or PurePosixPath(linkname).is_absolute() or "\\" in linkname:
        raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Invalid target for {name}: {linkname[:180]}")
    base = name.parent.as_posix() if symbolic else "."
    target = PurePosixPath(posixpath.normpath(posixpath.join(base, linkname)))
    if ".." in target.parts or target == PurePosixPath("."):
        raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Target escapes or points to the archive root: {name}")
    return target

def install_snapshot_links(destination, links, regular):
    # Ordinary files are extracted first, so archive contents cannot be written through a link.
    for name, entry in links.items():
        path = destination / name
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists() or path.is_symlink():
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Link collides with a staged file or directory: {name}")
        if entry["kind"] == "hardlink":
            target = entry["target"]
            visited = {name}
            while target in links and links[target]["kind"] == "hardlink" and target not in visited:
                visited.add(target)
                target = links[target]["target"]
            if target not in regular:
                raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Hardlink lacks an in-archive regular target: {name}")
            os.link(destination / target, path, follow_symlinks=False)
    for name, entry in links.items():
        if entry["kind"] == "symlink":
            path = destination / name
            if path.exists() or path.is_symlink():
                raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Symbolic link collides with a staged path: {name}")
            path.symlink_to(entry["linkname"])
    for name, entry in links.items():
        path = destination / name
        try:
            target = path.resolve(strict=True)
        except (OSError, RuntimeError) as exc:
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Broken or cyclic archive link: {name}") from exc
        if not target.is_relative_to(destination) or target in path.parents:
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Archive link target escapes or cycles to an ancestor: {name}")

def stage_public_snapshot(archive, destination, repo, deadline):
    destination = destination.resolve()
    destination.mkdir()
    total = 0
    regular, links = set(), {}
    with tarfile.open(archive, "r|gz") as tar:
        for member in tar:
            name = PurePosixPath(member.name)
            if name.is_absolute() or ".." in name.parts or "\\" in member.name:
                raise CPUHold("UNSAFE_SNAPSHOT_MEMBER", f"Unsafe archive path: {member.name[:200]}")
            remaining(deadline, POLICY["task_wall_seconds"])
            if member.issym() or member.islnk():
                if name in links or name in regular or name == PurePosixPath("."):
                    raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Duplicate or invalid archive link: {name}")
                links[name] = {
                    "kind": "symlink" if member.issym() else "hardlink",
                    "target": snapshot_link_target(name, member.linkname, member.issym()),
                    "linkname": member.linkname,
                }
                continue
            if not (member.isfile() or member.isdir()) or name in links or name in regular:
                raise CPUHold("UNSAFE_SNAPSHOT_MEMBER", f"Invalid or duplicate archive member: {name} ({member.type!r})")
            total += member.size
            if total > POLICY["snapshot_bytes_expanded_cap"]:
                raise CPUHold("SNAPSHOT_SIZE_LIMIT", "Expanded archive exceeded its CPU staging limit")
            tar.extract(member, destination, filter="data")
            if member.isfile():
                regular.add(name)
    install_snapshot_links(destination, links, regular)

    package, layouts = PACKAGE_LAYOUTS[repo]
    candidates = [
        (item.parent, layout) for item in destination.rglob(".git") if item.is_dir()
        for layout in layouts if (item.parent / layout / "__init__.py").is_file()
    ]
    if len(candidates) != 1:
        raise CPUHold("SNAPSHOT_LAYOUT", f"Expected one Git checkout containing {package}")
    root, package_dir = candidates[0]
    link_inventory = []
    for name, entry in sorted(links.items()):
        path = destination / name
        target = path if entry["kind"] == "symlink" else destination / entry["target"]
        if path.is_relative_to(root) and not target.resolve(strict=True).is_relative_to(root):
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Link in the checkout points outside it: {name}")
        link_inventory.append({"path": name.as_posix(), "kind": entry["kind"],
                               "target": entry["target"].as_posix()})
    return root, package, package_dir, link_inventory

def git_bytes(root, env, deadline, *argv):
    result = subprocess.run(
        ("git", "-C", str(root), *argv), env=env, capture_output=True,
        timeout=remaining(deadline, 20), check=False,
    )
    if result.returncode:
        raise CPUHold("NATIVE_GIT_PROBE", "Git could not read the mounted checkout identity")
    return result.stdout

def prepare_pytest_config(root):
    """Match the organizer subprocess harness's baseline pytest setup."""
    ini = root / "pytest.ini"
    text = ini.read_text(errors="replace") if ini.is_file() else ""
    if "# Hermetic test discovery" not in text and "[pytest]" not in text:
        ini.write_text(HERMETIC_INI, encoding="utf-8")
    conftest = root / "conftest.py"
    old = conftest.read_text(errors="replace") if conftest.is_file() else ""
    if not any(f"# {kind} test discovery hook for SWE-gemma" in old for kind in ("Hermetic", "Standard")):
        clean = old.replace(HERMETIC_HEADER, "").strip()
        conftest.write_text(HERMETIC_HEADER + ("\n" + clean if clean else "") + "\n", encoding="utf-8")

def visible_definition_paths(root):
    paths = set(RICH_OPTIONAL_TEST_DEFINITIONS)
    paths.update(name for name in CONFIG_FILES if (root / name).is_file())
    for path in (root / "tests").rglob("*"):
        relative = path.relative_to(root)
        if (any(part in EXCLUDED_DIRS for part in relative.parts)
                or path.name in EXCLUDED_NAMES or path.suffix in EXCLUDED_SUFFIXES):
            continue
        if path.is_file():
            paths.add(relative.as_posix())
    return tuple(sorted(paths))

def visible_definition_binding(root, files):
    regular, links = [], {}
    for name in files:
        path = root / name
        if not path.is_symlink():
            regular.append(name)
            continue
        target = path.resolve(strict=True)
        if not target.is_relative_to(root) or not target.is_file():
            raise CPUHold("UNSAFE_VISIBLE_DEFINITION_LINK", f"Invalid linked public fixture: {name}")
        links[name] = {"link_text": os.readlink(path), "target": target.relative_to(root).as_posix(),
                       "target_sha256": sha256_file(target)}
    digest = definition_files_fingerprint(root, regular, allow_missing=True)
    if links:
        digest = canonical_hash({"regular_sha256": digest, "internal_fixture_links": links})
    return digest, links

IMPORT_PROBE = """
import importlib
from pathlib import Path
import sys
module = importlib.import_module(sys.argv[1])
origin = Path(module.__file__).resolve()
print(origin)
if not origin.is_relative_to(Path(sys.argv[2]).resolve()):
    raise RuntimeError("Production package import escaped the public snapshot")
"""

## 5. Public pytest identity and evidence helpers

Reused from the pinned GP03 qualification notebook.

In [5]:
CAPTURE_PLUGIN = """import json
import os
from pathlib import Path
import sys

def pytest_collection_finish(session):
    missing = set()
    if os.environ["GP03_PUBLIC_PACKAGE"] == "requests":
        for item in session.items:
            fixture_info = getattr(item, "_fixtureinfo", None)
            if fixture_info is None:
                continue
            for name in {"httpbin", "httpbin_secure"}.intersection(item.fixturenames):
                definitions = fixture_info.name2fixturedefs.get(name) or ()
                # Requests may wrap the plugin fixture by requesting its own name.
                if not any(name not in definition.argnames for definition in definitions):
                    missing.add(name)
    capture = {
        "root": str(session.config.rootpath.resolve()),
        "nodeids": [item.nodeid for item in session.items],
        "item_files": [str(item.path.resolve()) for item in session.items],
        "missing_optional_fixtures": sorted(missing),
    }
    Path(os.environ["GP03_CAPTURE_PATH"]).write_text(json.dumps(capture), encoding="utf-8")

def pytest_sessionfinish(session, exitstatus):
    target = Path(os.environ["GP03_CAPTURE_PATH"])
    if target.is_file():
        capture = json.loads(target.read_text(encoding="utf-8"))
        package = sys.modules.get(os.environ["GP03_PUBLIC_PACKAGE"])
        capture["production_origin"] = getattr(package, "__file__", None)
        target.write_text(json.dumps(capture), encoding="utf-8")
"""

def diagnostic_text(output):
    fragments = []
    for name in ("stdout.txt", "stderr.txt"):
        path = output / name
        with path.open("rb") as stream:
            head = stream.read(32 * 1024)
            if path.stat().st_size > len(head):
                stream.seek(max(len(head), path.stat().st_size - 32 * 1024))
                head += b"\n" + stream.read()
        fragments.append(head.decode("utf-8", errors="replace"))
    return "\n".join(fragments)

def missing_imports(text):
    return set(re.findall(r"ModuleNotFoundError: No module named ['\"]([A-Za-z0-9_.]+)['\"]", text))

def read_node_capture(path, selector, root, package_dir):
    if not path.is_file():
        raise CPUHold("NODE_CAPTURE_MISSING", "pytest produced no independent node ID capture")
    capture = json.loads(path.read_text(encoding="utf-8"))
    nodes = capture.get("nodeids", [])
    if not nodes:
        raise CPUHold("ZERO_COLLECTION", "The selected public suite collected zero tests")
    if len(nodes) > POLICY["max_collected_nodes"]:
        raise CPUHold("COLLECTION_LIMIT", f"Collected {len(nodes)} nodes, exceeding the frozen CPU cap")
    pytest_root = Path(capture["root"]).resolve(strict=True)
    if not pytest_root.is_relative_to(root):
        raise CPUHold("COLLECTION_IDENTITY", "pytest root is outside the staged checkout")
    files = capture.get("item_files", [])
    if len(files) != len(nodes):
        raise CPUHold("COLLECTION_IDENTITY", "pytest omitted a collected file identity")
    canonical = []
    for node, item_file in zip(nodes, files):
        if not isinstance(node, str):
            raise CPUHold("COLLECTION_IDENTITY", "Invalid pytest node ID")
        filename, delimiter, suffix = node.partition("::")
        relative = PurePosixPath(filename)
        if not delimiter or not suffix or relative.is_absolute() or ".." in relative.parts or "\\" in filename:
            raise CPUHold("COLLECTION_IDENTITY", "Invalid relative pytest node path")
        path_from_node = pytest_root / relative
        if (not path_from_node.is_relative_to(root) or str(path_from_node.resolve(strict=True)) != item_file):
            raise CPUHold("COLLECTION_IDENTITY", "Node path disagrees with the file collected by pytest")
        canonical.append(path_from_node.relative_to(root).as_posix() + "::" + suffix)
    if "::" in selector:
        permitted = lambda item: item == selector or item.startswith(selector + "[")
    else:
        permitted = lambda item: item.startswith(selector + "::")
    if not all(permitted(n) for n in canonical) or len(canonical) != len(set(canonical)):
        raise CPUHold("COLLECTION_IDENTITY", "Canonical node identities differ from the selector")
    origin = capture.get("production_origin")
    if origin and not Path(origin).resolve().is_relative_to(root / package_dir):
        raise CPUHold("PACKAGE_ORIGIN_CHANGED", "pytest imported production code outside the snapshot")
    return canonical, capture["missing_optional_fixtures"]

def run_visible_phase(*, selector, root, package, package_dir, temporary, output,
                      deadline, baseline_sha, collect_only):
    argv = [sys.executable, "-m", "pytest", selector, "-q", "--tb=short",
            "-p", "no:cacheprovider", "-p", "_gp03_capture"]
    capture = output / "nodeids.json"
    if collect_only:
        argv.append("--collect-only")
        cap = POLICY["collect_wall_seconds"]
    else:
        argv.append(f"--junitxml={output / 'junit.xml'}")
        cap = POLICY["baseline_wall_seconds"]
    result = bounded_command(
        argv, root, command_env(root, temporary, capture, package),
        output, remaining(deadline, cap),
    )
    result["source_sha256_after"] = WorkspaceFingerprinter(root).snapshot().sha256
    write_json(output / "command.json", result)
    if result["source_sha256_after"] != baseline_sha:
        raise CPUHold("SOURCE_CHANGED", "The selected pytest command changed the staged source or test definitions")
    if result["timed_out"]:
        reason = "COLLECTION_TIMEOUT" if collect_only else "BASELINE_TIMEOUT"
        raise CPUHold(reason, "The selected public pytest command exceeded its CPU limit")
    if collect_only and result["exit_code"] == 5:
        raise CPUHold("ZERO_COLLECTION", "The selected public suite collected zero tests")
    return result, capture

def junit_counts(junit: Path | bytes) -> tuple[int, int, int, int, int]:
    """Extract actual testcase elements; never trust exit code or suite labels alone."""
    data = junit if isinstance(junit, bytes) else junit.read_bytes()
    root = ET.fromstring(data)
    cases = [el for el in root.iter() if el.tag.rsplit("}", 1)[-1] == "testcase"]
    collected = len(cases)
    fail = sum(any(c.tag.rsplit("}", 1)[-1] == "failure" for c in t) for t in cases)
    err = sum(any(c.tag.rsplit("}", 1)[-1] == "error" for c in t) for t in cases)
    skipped = sum(any(c.tag.rsplit("}", 1)[-1] == "skipped" for c in t) for t in cases)
    passed = collected - fail - err - skipped
    if passed < 0 or collected != fail + err + skipped + passed:
        raise ValueError("Contradictory JUnit testcase counts")
    return collected, passed, fail, err, skipped

def artifact_paths(folder, excluded=()):
    files = {}
    for path in sorted(folder.rglob("*")):
        relative = path.relative_to(folder).as_posix()
        if path.is_symlink():
            raise CPUHold("UNSAFE_EVIDENCE_PATH", f"Symlink in CPU evidence at {relative}")
        if path.is_file() and relative not in excluded:
            files[relative] = path
    return files

def seal_zip(output, files):
    """Write a new evidence archive and index every included file."""
    output = Path(output)
    index = {}
    temporary = output.with_suffix('.partial.zip')
    if output.exists() or temporary.exists():
        raise FileExistsError(f'Refusing to overwrite evidence {output}')
    try:
        with zipfile.ZipFile(temporary, 'x', compression=zipfile.ZIP_DEFLATED) as z:
            for name, path in sorted(files.items()):
                path = Path(path)
                if path.is_symlink() or not path.is_file():
                    raise ValueError(f'Unsafe evidence file {name}')
                index[name] = {'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
                z.write(path, name)
            z.writestr('SHA256_MANIFEST.json', json.dumps(index, indent=2, sort_keys=True) + '\n')
        verify_zip(temporary)
        temporary.rename(output)
    finally:
        temporary.unlink(missing_ok=True)
    print(f'Evidence ZIP {output.name} | SHA-256 {sha256_file(output)}', flush=True)
    return output

def verify_zip(archive):
    with zipfile.ZipFile(archive) as z:
        names = z.namelist()
        index = json.loads(z.read('SHA256_MANIFEST.json'))
        if len(names) != len(set(names)) or set(names) != set(index) | {'SHA256_MANIFEST.json'}:
            raise RuntimeError('ZIP member set disagrees with the checksum manifest')
        for name, pin in index.items():
            path = Path(name)
            if path.is_absolute() or '..' in path.parts or z.getinfo(name).file_size != pin['bytes']:
                raise RuntimeError(f'Invalid archive member {name}')
            digest = hashlib.sha256()
            with z.open(name) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(block)
            if digest.hexdigest() != pin['sha256']:
                raise RuntimeError(f'Archive checksum mismatch for {name}')
    return index

def offer_zip_download(bundle, auto_download=True, embed_limit_mb=10):
    from pathlib import Path
    import base64
    import html
    import json
    import os
    from IPython.display import FileLink, HTML, Javascript, display

    bundle = Path(bundle).resolve()
    relative = os.path.relpath(bundle, Path.cwd())
    print('ZIP ready:', bundle)
    print('If the browser does not start a download, use the Download ZIP button or Kaggle Output panel.')
    display(FileLink(relative))
    if bundle.stat().st_size > embed_limit_mb * 1024 * 1024:
        print('This ZIP exceeds the inline-download size limit; use the file link or Output panel.')
        return {'automatic_download_requested': False, 'download_confirmed': False, 'reason': 'size_limit'}

    encoded = base64.b64encode(bundle.read_bytes()).decode('ascii')
    name = html.escape(bundle.name, quote=True)
    button = ('<a download="' + name + '" href="data:application/zip;base64,' + encoded + '" '
              'style="display:inline-block;padding:12px 20px;background:#174EA6;color:white;'
              'text-decoration:none;border-radius:6px;font-weight:600">Download ZIP</a>')
    display(HTML(button))
    if auto_download:
        script = '''(function () {
  try {
    const payload = __PAYLOAD__;
    const bytes = Uint8Array.from(atob(payload), function (c) { return c.charCodeAt(0); });
    const url = URL.createObjectURL(new Blob([bytes], {type: "application/zip"}));
    const link = document.createElement("a");
    link.href = url;
    link.download = __FILENAME__;
    link.style.display = "none";
    document.body.appendChild(link);
    link.click();
    link.remove();
    setTimeout(function () { URL.revokeObjectURL(url); }, 60000);
  } catch (error) {
    console.info("Automatic ZIP download was not available; use the Download ZIP button or Output panel.", error);
  }
})();'''.replace('__PAYLOAD__', json.dumps(encoded)).replace('__FILENAME__', json.dumps(bundle.name))
        display(Javascript(script))
        print('Automatic browser-download request offered. Notebook code cannot confirm that your PC saved it.')
    return {'automatic_download_requested': bool(auto_download), 'download_confirmed': False,
            'fallbacks': ['Download ZIP button', 'file link', 'Kaggle Output panel']}

## 6. Shared verification contracts

Reuse the controller's check, observation and deadline contracts.

In [6]:
# Reused from the pinned GP02C-04B source gp02c_ledger.py.
ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,120}$")

@dataclass(frozen=True)
class VisibleCheck:
    check_id: str
    command: tuple[str, ...]
    definition_sha256: str
    reserve_seconds: float = 0.0
    origin: str = "visible_repo"
    working_dir: str = "."
    test_selection: str = ""
    expected_collected: int | None = None
    pinned_environment_sha256: str | None = None
    definition_files: tuple[str, ...] = ()
    definition_files_sha256: str | None = None
    allow_missing_definition_files: bool = False

    def __post_init__(self):
        if not ID_RE.fullmatch(self.check_id):
            raise ValueError("Unsafe visible check identifier")
        if self.origin != "visible_repo":
            raise ValueError("Only trusted visible-repository checks are allowed")
        if not self.command or not all(isinstance(x, str) and x for x in self.command):
            raise ValueError("Nonempty command tokens required")
        if not re.fullmatch(r"[0-9a-f]{64}", self.definition_sha256):
            raise ValueError("Check definition must have SHA-256 identity")
        if self.reserve_seconds < 0:
            raise ValueError("Negative estimate")
        if (Path(self.working_dir).is_absolute() or
            any(x == ".." for x in Path(self.working_dir).parts)):
            raise ValueError("Working directory must be workspace-relative")
        if self.expected_collected is not None and self.expected_collected < 1:
            raise ValueError("Expected collection must be positive")
        if self.pinned_environment_sha256 is not None and not re.fullmatch(
            r"[0-9a-f]{64}", self.pinned_environment_sha256):
            raise ValueError("Bad pinned environment fingerprint")
        if bool(self.definition_files) != bool(self.definition_files_sha256):
            raise ValueError("Pinned visible definition files and digest must be supplied together")
        if self.definition_files_sha256 and not re.fullmatch(r"[0-9a-f]{64}", self.definition_files_sha256):
            raise ValueError("Bad pinned visible definition content fingerprint")
        for path in self.definition_files:
            rel = Path(path)
            if (not path or rel.is_absolute() or ".." in rel.parts or rel.as_posix() != path):
                raise ValueError("Unsafe visible definition filename")

    @property
    def command_sha256(self) -> str:
        return canonical_hash(self.command)

@dataclass(frozen=True)
class TestObservation:
    exit_code: int | None
    tests_collected: int | None
    tests_passed: int | None
    tests_failed: int | None
    tests_errored: int | None
    elapsed_seconds: float
    stdout: str = ""
    stderr: str = ""
    timed_out: bool = False
    setup_error: bool = False
    tests_skipped: int | None = 0

    def __post_init__(self):
        if self.elapsed_seconds < 0:
            raise ValueError("Negative duration")
        for name in ("tests_collected", "tests_passed", "tests_failed", "tests_errored", "tests_skipped"):
            v = getattr(self, name)
            if v is not None and v < 0:
                raise ValueError(f"Negative test count: {name}")

    @property
    def observed_outcome(self) -> str:
        if self.timed_out or self.setup_error or self.exit_code is None:
            return "ERROR"
        counts = (self.tests_collected, self.tests_passed,
                  self.tests_failed, self.tests_errored, self.tests_skipped)
        if any(c is None for c in counts):
            return "ERROR"
        total, passed, failed, errored, skipped = counts
        if total < 1 or passed + failed + errored + skipped != total:
            return "ERROR"
        if errored > 0:
            return "ERROR"
        if failed > 0 and self.exit_code != 0:
            return "FAIL"
        if failed > 0 and self.exit_code == 0:
            return "ERROR"  # contradictory successful process
        if self.exit_code != 0:
            return "ERROR"  # e.g. missing dependency / collection failure
        if passed < 1:
            return "ERROR"  # all skipped / no executed positive check
        return "PASS"

# Reused from the pinned GP02C-04B source gp02c_live_adapter.py.
@dataclass
class RealDeadline:
    """One absolute clock per attempt. 840/900s defaults; no reset per check."""
    repair_cutoff_seconds: float = 840.0
    overall_cutoff_seconds: float = 900.0
    started_monotonic: float | None = None
    now: Callable[[], float] = time.monotonic

    def __post_init__(self):
        if not 0 < self.repair_cutoff_seconds < self.overall_cutoff_seconds:
            raise ValueError("Need 0 < repair cutoff < overall cutoff")
        if self.started_monotonic is None:
            self.started_monotonic = self.now()

    def remaining(self, phase: str) -> float:
        if phase not in {"ordinary", "proactive", "final"}:
            raise ValueError("Unrecognized execution phase")
        cap = self.overall_cutoff_seconds if phase == "final" else self.repair_cutoff_seconds
        return max(0.0, self.started_monotonic + cap - self.now())

    def elapsed(self) -> float:
        return self.now() - self.started_monotonic

def _read_head(path: Path, max_bytes: int = 6000) -> str:
    with path.open("rb") as f:
        return f.read(max_bytes).decode("utf-8", errors="replace")

## 7. Public adapter

Verify an isolated copy and record the result under the shared clock.

In [7]:
def copy_gp03_workspace(source, destination):
    """Copy the public working tree and verify all links in the private copy."""
    source = source.resolve(strict=True)
    before = WorkspaceFingerprinter(source).snapshot()

    def ignore_generated(directory, names):
        ignored = set()
        for name in names:
            if Path(directory, name).is_dir():
                if name in EXCLUDED_DIRS:
                    ignored.add(name)
            elif name in EXCLUDED_NAMES or Path(name).suffix in EXCLUDED_SUFFIXES:
                ignored.add(name)
        return ignored

    shutil.copytree(source, destination, symlinks=True, ignore=ignore_generated)
    copied = WorkspaceFingerprinter(destination).snapshot()
    if copied.sha256 != before.sha256:
        raise CPUHold("ISOLATED_COPY_MISMATCH", "Public source fingerprint changed during copying")
    links = []
    for name, kind, _ in copied.entries:
        if kind != "L":
            continue
        link = destination / name
        target_text = os.readlink(link)
        snapshot_link_target(PurePosixPath(name), target_text, symbolic=True)
        try:
            target = link.resolve(strict=True)
        except (OSError, RuntimeError) as exc:
            raise CPUHold("UNSAFE_ISOLATED_LINK", f"Broken or cyclic public link {name}") from exc
        if (not target.is_relative_to(destination) or target == destination
                or (target.is_dir() and link.parent.is_relative_to(target))
                or not (target.is_dir() or target.is_file())):
            raise CPUHold("UNSAFE_ISOLATED_LINK", f"Unsafe public link {name}")
        links.append({"path": name, "target": target.relative_to(destination).as_posix()})
    return before.sha256, links


def gp03_check_from_manifest(manifest):
    fields = dict(manifest["check"])
    fields["command"] = tuple(fields["command"])
    fields["definition_files"] = tuple(fields["definition_files"])
    return VisibleCheck(**fields)


class GP03PublicPytestAdapter:
    """Execute one pinned public check and return the controller's TestObservation."""

    def __init__(self, *, source_root, manifest, environment_sha256, deadline, results_root):
        self.source_root = Path(source_root).resolve(strict=True)
        self.results_root = Path(results_root).resolve()
        if (self.results_root.is_relative_to(self.source_root)
                or self.source_root.is_relative_to(self.results_root)):
            raise ValueError("Evidence and source must be in separate directories")
        self.results_root.mkdir(parents=True, exist_ok=True)
        self.manifest = manifest
        self.check = gp03_check_from_manifest(manifest)
        self.environment_sha256 = environment_sha256
        self.deadline = deadline
        self.phase = "ordinary"
        self.executions = []

    def _validate_definition(self, check, root):
        selector = check.test_selection
        if (check != self.check or check.working_dir != "."
                or check.pinned_environment_sha256 != self.environment_sha256
                or check.command != (sys.executable, "-m", "pytest", selector, "-q", "--tb=short")
                or not PUBLIC_TEST.fullmatch(selector.partition("::")[0])):
            raise CPUHold("CHECK_PIN_MISMATCH", "The trusted public command differs from its pin")
        files = visible_definition_paths(root)
        if files != check.definition_files:
            raise CPUHold("VISIBLE_DEFINITION_SET_CHANGED", "The public test or configuration file set changed")
        definition_sha, fixture_links = visible_definition_binding(root, files)
        if (definition_sha != check.definition_files_sha256
                or fixture_links != self.manifest.get("definition_link_targets", {})):
            raise CPUHold("VISIBLE_DEFINITION_CHANGED", "The public test, configuration or fixture bytes changed")

    def __call__(self, check):
        started = time.monotonic()
        absolute_deadline = started + self.deadline.remaining(self.phase)
        folder = self.results_root / f"check_{len(self.executions) + 1:04d}"
        folder.mkdir()
        receipt = {"check_id": check.check_id, "phase": self.phase, "reason": None}
        phase = folder / "pytest"
        counts = (None,) * 5
        exit_code = None
        try:
            remaining(absolute_deadline, POLICY["baseline_wall_seconds"])
            with tempfile.TemporaryDirectory(prefix="gp03_isolated_", dir=self.results_root.parent) as work:
                temporary = Path(work).resolve()
                isolated = temporary / "checkout"
                source_sha, links = copy_gp03_workspace(self.source_root, isolated)
                receipt.update(source_sha256_before=source_sha, isolated_links=links)
                self._validate_definition(check, isolated)
                package = PACKAGE_LAYOUTS[self.manifest["repo"]][0]
                package_dir = self.manifest["production_root"].rstrip("/")
                (temporary / "_gp03_capture.py").write_text(CAPTURE_PLUGIN, encoding="utf-8")
                result, capture = run_visible_phase(
                    selector=check.test_selection, root=isolated, package=package, package_dir=package_dir,
                    temporary=temporary, output=phase, deadline=absolute_deadline,
                    baseline_sha=source_sha, collect_only=False,
                )
                exit_code = result["exit_code"]
                nodes, missing = read_node_capture(capture, check.test_selection, isolated, package_dir)
                junit = phase / "junit.xml"
                counts = junit_counts(junit)
                receipt.update(actual_nodeids_sha256=canonical_hash(nodes), junit_sha256=sha256_file(junit),
                               isolated_source_sha256_after=result["source_sha256_after"])
                total, passed, failed, errors, skipped = counts
                if missing or nodes != self.manifest["expected_nodeids"] or total != check.expected_collected:
                    raise CPUHold("CHECK_NODE_MISMATCH", "Public check collection differs from its frozen node IDs")
                if errors or total == skipped or (exit_code, bool(failed)) not in {(0, False), (1, True)}:
                    raise CPUHold("PUBLIC_PYTEST_ERROR", "JUnit and the pytest exit status do not show a complete check")
                if failed and missing_imports(diagnostic_text(phase)):
                    raise CPUHold("PUBLIC_PYTEST_DEPENDENCY", "A completed test raised a missing dependency error")
                remaining(absolute_deadline, 1)
        except CPUHold as exc:
            receipt.update(reason=exc.reason, detail=str(exc))
        except (OSError, ValueError, ET.ParseError) as exc:
            receipt.update(reason="PUBLIC_CHECK_ERROR", detail=f"{type(exc).__name__}: {exc}")
        finally:
            if "source_sha256_before" in receipt:
                after = WorkspaceFingerprinter(self.source_root).snapshot().sha256
                receipt["source_sha256_after"] = after
                if after != receipt["source_sha256_before"]:
                    receipt.update(reason="ORIGINAL_SOURCE_CHANGED", detail="Source changed while pytest ran")
        total, passed, failed, errors, skipped = counts
        stdout = _read_head(phase / "stdout.txt") if (phase / "stdout.txt").is_file() else ""
        stderr = _read_head(phase / "stderr.txt") if (phase / "stderr.txt").is_file() else ""
        if not receipt["reason"] and time.monotonic() >= absolute_deadline:
            receipt.update(reason="TASK_DEADLINE", detail="The public check exceeded its phase deadline")
        if receipt["reason"]:
            stderr += f"\n{receipt['reason']} | {receipt['detail']}"
        observation = TestObservation(
            exit_code, total, passed, failed, errors, round(time.monotonic() - started, 3),
            stdout=stdout, stderr=stderr,
            timed_out=receipt["reason"] in {"TASK_DEADLINE", "BASELINE_TIMEOUT"},
            setup_error=receipt["reason"] is not None, tests_skipped=skipped,
        )
        receipt.update(outcome=observation.observed_outcome, observation=asdict(observation))
        write_json(folder / "Public_Check_Receipt.json", receipt)
        self.executions.append(receipt)
        return observation

## 8. Acceptance checks

Compare the adapter's observation against the verified CPU receipt for each case.

In [8]:
def load_acceptance_pins(working, explicit_path, evidence, plan):
    if explicit_path is not None:
        path = Path(explicit_path)
    else:
        path = working / evidence["filename"]
        if not path.is_file():
            matches = sorted(Path("/kaggle/input").rglob(evidence["filename"]))
            path = next((match for match in matches if sha256_file(match) == evidence["sha256"]), path)
    if not path.is_file():
        raise CPUHold("CPU_EVIDENCE_MISSING", f"Place {evidence['filename']} in /kaggle/working or mount it as an input")
    if sha256_file(path) != evidence["sha256"]:
        raise CPUHold("CPU_EVIDENCE_MISMATCH", "The qualification evidence ZIP differs from its SHA-256 pin")
    index = verify_zip(path)
    pins = {}
    with zipfile.ZipFile(path) as archive:
        report = json.loads(archive.read("GP03_CPU_Report.json"))
        if (report["run_sha256"] != evidence["qualification_run_sha256"]
                or report["notebook_recipe_sha256"] != evidence["notebook_recipe_sha256"]
                or report["tasks_sha256"] != evidence["tasks_sha256"]
                or report["gp00_split_sha256"] != evidence["gp00_split_sha256"]
                or environment_fingerprint(report["environment"]) != evidence["cpu_environment_sha256"]
                or report["environment_sha256"] != evidence["cpu_environment_sha256"]
                or report["evaluation_frozen"] or report["model_calls"] or report["protected_verifier_calls"]):
            raise CPUHold("CPU_REPORT_MISMATCH", "The qualification report disagrees with the verified public evidence")
        rows = {row["task_id"]: row for row in report["rows"]}
        for entry in plan:
            task = entry["task_id"]
            prefix = f"tasks/{task}/"
            manifest_name = prefix + "CPU_Check_Manifest.json"
            receipt_name = prefix + "Task_Receipt.json"
            result_name = prefix + "Task_Result.json"
            junit_name = prefix + "baseline/junit.xml"
            manifest = json.loads(archive.read(manifest_name))
            receipt = json.loads(archive.read(receipt_name))
            prior = json.loads(archive.read(result_name))
            expected = junit_counts(archive.read(junit_name))
            run_sha = (report["prior_evidence"]["run_sha256"] if entry["cpu_receipt_origin"] == "v1.0_retained"
                       else report["run_sha256"])
            check = gp03_check_from_manifest(manifest)
            same_identity = all(manifest[k] == entry[k] for k in (
                "task_id", "repo", "base_commit", "native_head", "native_tree", "snapshot_sha256", "baseline_outcome",
            ))
            if (not same_identity or manifest_name != entry["cpu_check_member"]
                    or index[manifest_name]["sha256"] != entry["cpu_check_sha256"]
                    or index[receipt_name]["sha256"] != entry["cpu_receipt_sha256"]
                    or receipt["task_id"] != task or receipt["run_sha256"] != run_sha
                    or receipt["files"] != {k.removeprefix(prefix): v for k, v in index.items()
                                             if k.startswith(prefix) and k != receipt_name}
                    or prior != rows[task] or prior["status"] != "CPU_READY"
                    or manifest["schema"] != "gp03.public-cpu-check.v1"
                    or manifest["environment_sha256"] != evidence["cpu_environment_sha256"]
                    or manifest["source_policy"] != "gp02c_workspace_all_files_excluding_fixed_generated_artifacts"
                    or manifest["baseline_junit_sha256"] != index[junit_name]["sha256"]
                    or manifest["expected_nodeids_sha256"] != canonical_hash(manifest["expected_nodeids"])
                    or list(check.command) != entry["public_check_argv"]
                    or check.expected_collected != entry["expected_collected"]
                    or check.expected_collected != len(manifest["expected_nodeids"])
                    or check.expected_collected != expected[0]):
                raise CPUHold("TASK_RECEIPT_MISMATCH", f"Qualification receipt or check differs from its pins for {task}")
            link_file = prefix + "Snapshot_Links.json"
            links = json.loads(archive.read(link_file)) if link_file in index else []
            pins[task] = (manifest, expected, links)
    return pins


def accept_gp03_task(entry, pin, competition, work, output, environment_sha):
    manifest, expected_counts, expected_links = pin
    task = entry["task_id"]
    row = {"task_id": task, "pair_index": entry["pair_index"], "expected_collected": expected_counts[0],
           "actual_collected": None, "status": "ADAPTER_HOLD", "outcome": "NOT_RUN", "reason": None}
    started = time.monotonic()
    staging_deadline = started + POLICY["task_wall_seconds"]
    output.mkdir(parents=True)
    try:
        archive = competition / "snapshots" / f"{task}.tgz"
        if (not archive.is_file() or archive.stat().st_size != entry["snapshot_bytes"]
                or sha256_file(archive) != entry["snapshot_sha256"]):
            raise CPUHold("SNAPSHOT_MISMATCH", "Expected the exact published snapshot from the qualification run")
        with tempfile.TemporaryDirectory(prefix=f"gp03_{task}_", dir=work) as scratch:
            temporary = Path(scratch).resolve()
            root, package, package_dir, links = stage_public_snapshot(
                archive, temporary / "staged", entry["repo"], staging_deadline,
            )
            env = command_env(root, temporary)
            for key, ref in (("native_head", "HEAD^{commit}"), ("native_tree", "HEAD^{tree}")):
                observed = git_bytes(root, env, staging_deadline, "rev-parse", "--verify", ref).decode().strip()
                row[key] = observed
                if observed != entry[key]:
                    raise CPUHold("NATIVE_GIT_IDENTITY", f"The staged Git {key} differs from qualification")
            if links != expected_links or package_dir + "/" != manifest["production_root"]:
                raise CPUHold("SNAPSHOT_LAYOUT_MISMATCH", "The public package or link inventory differs from qualification")
            prepare_pytest_config(root)
            source_sha = WorkspaceFingerprinter(root).snapshot().sha256
            if source_sha != manifest["baseline_source_sha256"]:
                raise CPUHold("BASELINE_SOURCE_MISMATCH", "The staged public source differs from qualification")
            row["staging_seconds"] = round(time.monotonic() - started, 3)
            probe = bounded_command(
                [sys.executable, "-c", IMPORT_PROBE, package, str(root / package_dir)],
                root, env, output / "package_origin", remaining(staging_deadline, 30),
            )
            if probe["exit_code"] or probe["timed_out"]:
                raise CPUHold("PACKAGE_IMPORT_HOLD", "The public package import did not resolve to the snapshot")
            adapter = GP03PublicPytestAdapter(
                source_root=root, manifest=manifest, environment_sha256=environment_sha,
                deadline=RealDeadline(POLICY["baseline_wall_seconds"], POLICY["final_reserve_seconds"]),
                results_root=output / "checks",
            )
            adapter.phase = "final"
            observation = adapter(adapter.check)
            observed = adapter.executions[-1]
            counts = (observation.tests_collected, observation.tests_passed, observation.tests_failed,
                      observation.tests_errored, observation.tests_skipped)
            row.update(actual_collected=counts[0], outcome=observation.observed_outcome,
                       passed=counts[1], failed=counts[2], errors=counts[3], skipped=counts[4],
                       check_seconds=observation.elapsed_seconds, isolated_links=len(observed.get("isolated_links", [])),
                       expected_nodeids_sha256=manifest["expected_nodeids_sha256"],
                       actual_nodeids_sha256=observed.get("actual_nodeids_sha256"))
            if observed["reason"]:
                raise CPUHold(observed["reason"], observed["detail"])
            if observed["source_sha256_before"] != source_sha:
                raise CPUHold("BASELINE_SOURCE_MISMATCH", "The import probe changed the staged source")
            if counts != expected_counts or observation.observed_outcome != manifest["baseline_outcome"]:
                raise CPUHold("BASELINE_OBSERVATION_MISMATCH", "Replayed JUnit counts or outcome differ from qualification")
            row["status"] = "ADAPTER_READY"
    except CPUHold as exc:
        row.update(reason=exc.reason, detail=str(exc))
    except (OSError, ValueError, tarfile.TarError, subprocess.SubprocessError) as exc:
        row.update(reason="PUBLIC_ADAPTER_RUNTIME_ERROR", detail=f"{type(exc).__name__}: {exc}")
    row["elapsed_seconds"] = round(time.monotonic() - started, 3)
    write_json(output / "Adapter_Acceptance.json", row)
    return row


def run_gp03_acceptance(competition, working, cpu_zip, evidence, plan, development):
    started = time.monotonic()
    working = working.resolve()
    working.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    name = f"GP03_Public_Adapter_Acceptance_{stamp}_{uuid.uuid4().hex[:8]}"
    report = {"schema": "gp03.public-adapter-acceptance.v1", "run_id": name,
              "adapter_recipe_sha256": ADAPTER_RECIPE_SHA256,
              "eligible_pool_sha256": ELIGIBLE_POOL_SHA256,
              "qualification_evidence_sha256": evidence["sha256"], "model_calls": 0,
              "protected_verifier_calls": 0, "held_out_agent_attempts": 0, "evaluation_frozen": False,
              "planned": len(plan), "rows": [], "preflight_hold": None}
    with tempfile.TemporaryDirectory(prefix=".gp03_acceptance_", dir=working) as scratch:
        output = Path(scratch).resolve()
        try:
            if any(Path("/dev").glob("nvidia[0-9]*")):
                raise CPUHold("CPU_REQUIRED", "Set Accelerator to None before running this notebook")
            print(f"GP03 public adapter | {len(plan)} proposed tasks | CPU | model calls 0", flush=True)
            profile = environment_profile()
            environment_sha = environment_fingerprint(profile)
            report["environment_sha256"] = environment_sha
            write_json(output / "CPU_Environment.json", profile)
            if environment_sha != evidence["cpu_environment_sha256"]:
                raise CPUHold("CPU_ENVIRONMENT_MISMATCH", "Use the CPU image that produced GP03 qualification v1.1")
            load_gp03_inputs(competition, plan, development, evidence["tasks_sha256"])
            pins = load_acceptance_pins(working, cpu_zip, evidence, plan)
            print(f"Qualification evidence and task release verified | elapsed {time.monotonic() - started:.1f}s", flush=True)
            for entry in plan:
                task = entry["task_id"]
                prefix = f"[{entry['pair_index']:02}/{len(plan):02}] {task}"
                print(f"{prefix} staging public snapshot", flush=True)
                row = accept_gp03_task(entry, pins[task], competition, output,
                                       output / "tasks" / task, environment_sha)
                report["rows"].append(row)
                print(f"{prefix} {row['status']} | {row['actual_collected']}/{row['expected_collected']} tests"
                      f" | {row['reason'] or row['outcome']} | {row['elapsed_seconds']:.1f}s", flush=True)
        except CPUHold as exc:
            report["preflight_hold"] = {"reason": exc.reason, "detail": str(exc)}
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, ET.ParseError) as exc:
            report["preflight_hold"] = {"reason": "ACCEPTANCE_PREFLIGHT_ERROR", "detail": f"{type(exc).__name__}: {exc}"}
        ready = sum(row["status"] == "ADAPTER_READY" for row in report["rows"])
        report.update(ready=ready, holds=len(report["rows"]) - ready, pending=len(plan) - len(report["rows"]),
                      status="GP03_PUBLIC_ADAPTER_ACCEPTED" if ready == len(plan) else "GP03_PUBLIC_ADAPTER_HOLD",
                      elapsed_seconds=round(time.monotonic() - started, 3))
        if report["preflight_hold"]:
            print("Preflight hold | " + json.dumps(report["preflight_hold"]), flush=True)
        write_json(output / "GP03_Public_Adapter_Report.json", report)
        write_json(output / "GP03_Proposed_Pair_Order.json", {"evaluation_frozen": False, "pairs": plan})
        bundle = seal_zip(working / (name + ".zip"), artifact_paths(output))
    print(f"{report['status']} | {ready}/{len(plan)} accepted | {report['holds']} holds | {report['pending']} pending"
          f" | model calls 0 | elapsed {report['elapsed_seconds']:.1f}s", flush=True)
    return report, bundle

## 9. Run and download evidence

In [9]:
acceptance_report, evidence_zip = run_gp03_acceptance(
    COMPETITION_DIR, WORK_DIR, CPU_EVIDENCE_ZIP, GP03_CPU_EVIDENCE, GP03_PROPOSED_PAIRS, GP03_DEVELOPMENT,
)
offer_zip_download(evidence_zip, auto_download=AUTO_DOWNLOAD_ZIP)

GP03 public adapter | 9 proposed tasks | CPU | model calls 0
Qualification evidence and task release verified | elapsed 0.4s
[01/09] fastapi_14964 staging public snapshot
[01/09] fastapi_14964 ADAPTER_READY | 1/1 tests | PASS | 11.1s
[02/09] requests_7502 staging public snapshot
[02/09] requests_7502 ADAPTER_READY | 1/1 tests | PASS | 2.6s
[03/09] rich_3718 staging public snapshot
[03/09] rich_3718 ADAPTER_READY | 11/11 tests | PASS | 3.5s
[04/09] fastapi_12942 staging public snapshot
[04/09] fastapi_12942 ADAPTER_READY | 38/38 tests | PASS | 9.2s
[05/09] requests_7205 staging public snapshot
[05/09] requests_7205 ADAPTER_READY | 1/1 tests | PASS | 2.3s
[06/09] rich_4077 staging public snapshot
[06/09] rich_4077 ADAPTER_READY | 3/3 tests | PASS | 3.0s
[07/09] fastapi_15023 staging public snapshot
[07/09] fastapi_15023 ADAPTER_READY | 7/7 tests | PASS | 9.9s
[08/09] requests_7427 staging public snapshot
[08/09] requests_7427 ADAPTER_READY | 2/2 tests | PASS | 2.4s
[09/09] rich_3043 stag

/kaggle/working/GP03_Public_Adapter_Acceptance_20261109T002001Z_1c73a0b4.zip

<IPython.core.display.HTML object>

<IPython.core.display.Javascript object>

Automatic browser-download request offered. Notebook code cannot confirm that your PC saved it.


{'automatic_download_requested': True,
 'download_confirmed': False,
 'fallbacks': ['Download ZIP button', 'file link', 'Kaggle Output panel']}